# Sugidanon — Code-Switch Hiligaynon ASR Benchmark

One-click test: download the published dataset, run a Whisper baseline,
score the **switch-region WER** (the switch penalty).

Dataset: https://huggingface.co/datasets/LauelKills/sugidanon-hil-codeswitch  
Code: https://github.com/Jazztinn/tinig-sa-liwanag

**Runtime → Run all.** ~3-5 min on CPU.


## 1. Install


In [ ]:
!pip install -q openai-whisper huggingface_hub datasets
!apt-get -qq install -y ffmpeg >/dev/null


## 2. Get the code + dataset


In [ ]:
!git clone -q https://github.com/Jazztinn/tinig-sa-liwanag.git
%cd tinig-sa-liwanag
from huggingface_hub import snapshot_download
DS = snapshot_download('LauelKills/sugidanon-hil-codeswitch', repo_type='dataset')
print('dataset at:', DS)


## 3. Run the Whisper baseline
Whisper has no native Hiligaynon; `tl` (Tagalog) is the closest — that's the point.


In [ ]:
!python scripts/run_whisper.py --audio-dir $DS/data/audio --out-dir /content/preds --model small --language tl


## 4. Score — overall / switch-region / monolingual WER + switch penalty


In [ ]:
!python score.py --ref $DS/data/annotations --hyp /content/preds


## Result

A negative **switch penalty** is the finding: an off-the-shelf Tagalog model
transcribes the borrowed English/Tagalog switch words better than the
Hiligaynon matrix it was never trained on — the gap Sugidanon exposes.
Swap `--model small` for `large-v3`, or run Meta MMS, to compare models.


## (Optional) Developer one-liner — `load_dataset`
Audiofolder layout, so the dataset loads in one line. If this cell errors,
ignore it — the benchmark above is independent.


In [ ]:
from datasets import load_dataset
ds = load_dataset('LauelKills/sugidanon-hil-codeswitch', data_dir='data/audio', split='train')
print(ds)
print(ds[0]['transcript'], '|', ds[0]['switch_type'])
